# Python com SQLite
## Banco de Dados Relacional Sem Instalação

[◀ Anterior](33_DSA_Busca_Ordenacao.ipynb) | [📖 Índice](00_Index.ipynb) | [Próximo ▶](35_MongoDB.ipynb)

---

## 🎯 Objetivos

- Entender o que é um banco de dados relacional
- Conectar e criar bancos SQLite com `sqlite3` (já vem com Python!)
- Criar tabelas e inserir dados
- Consultar dados com filtros, ordenação e paginação
- Atualizar e deletar registros
- Relacionar tabelas com JOIN
- Prevenir SQL Injection com parâmetros preparados

## 📝 Introdução

Imagine que você tem uma planilha com milhares de linhas. Agora imagine que você quer buscar, filtrar e combinar dados de várias planilhas ao mesmo tempo. Um **banco de dados relacional** faz exatamente isso — mas de forma muito mais rápida e segura.

**SQLite** é um banco de dados relacional que não precisa de servidor, não precisa de instalação e salva tudo em **um único arquivo** `.db`. Ele vem junto com o Python, no módulo `sqlite3` — você só precisa importar e começar.

> 💡 **Vantagem para iniciantes:** Diferente do MySQL (que exige instalar e configurar um servidor), com SQLite você abre o notebook, importa e já funciona. Zero configuração.

## 📚 Explicação Detalhada

### O que é um banco de dados?

Um banco organiza dados em **tabelas**, como uma planilha:

| Tabela `alunos` | | | |
|---|---|---|---|
| **id** | **nome** | **idade** | **curso** |
| 1 | Maria | 20 | Ciência da Computação |
| 2 | João | 22 | Sistemas de Informação |

Cada **linha** é um registro, cada **coluna** é um campo.

### Conectando ao SQLite

```python
import sqlite3

# Conecta (cria o arquivo se não existir)
conexao = sqlite3.connect("escola.db")
cursor = conexao.cursor()

# Executa comandos SQL
cursor.execute("SELECT * FROM alunos")

# Fecha no final
cursor.close()
conexao.close()
```

O arquivo `escola.db` será criado na mesma pasta do notebook.

### Parâmetros preparados (segurança)

Para evitar SQL Injection, use `?` no lugar dos valores:

```python
# SEGURO:
cursor.execute("SELECT * FROM alunos WHERE nome = ?", (nome,))

# PERIGOSO (NUNCA faça isso):
cursor.execute(f"SELECT * FROM alunos WHERE nome = '{nome}'")
```

## 💻 Exemplos

> 🤔 **Pense:** Como o Python guarda dados na memória? Se o programa fecha, tudo se perde. Como fazer os dados sobreviverem ao fim do programa e poderem ser consultados depois?

In [ ]:
# Exemplo 1: Conectando e criando um banco de dados
import sqlite3

# O connect() já cria o banco se ele não existir
conexao = sqlite3.connect("escola.db")
cursor = conexao.cursor()

print("Banco 'escola.db' criado/conectado com sucesso!")
print("O arquivo está na mesma pasta deste notebook.")

cursor.close()
conexao.close()

> 🤔 **Pense:** Se o `connect()` cria o arquivo sozinho, o que acontece se você conectar com o nome errado? Será que cria um banco vazio sem você perceber?

In [ ]:
# Exemplo 2: Criando tabela
conexao = sqlite3.connect("escola.db")
cursor = conexao.cursor()

# SQLite não precisa de AUTO_INCREMENT — INTEGER PRIMARY KEY já faz isso
cursor.execute("""
    CREATE TABLE IF NOT EXISTS alunos (
        id INTEGER PRIMARY KEY,
        nome TEXT NOT NULL,
        idade INTEGER,
        curso TEXT,
        nota REAL
    )
""")
print("Tabela 'alunos' criada/verificada com sucesso!")

# Mostrar estrutura da tabela
cursor.execute("PRAGMA table_info(alunos)")
print("\nEstrutura da tabela:")
for coluna in cursor.fetchall():
    print(f"  {coluna[1]:<15s} {coluna[2]:<10s} obrigatório={bool(coluna[3])}")

cursor.close()
conexao.close()

> 🚨 **Erro comum:** Esquecer de fechar a conexão! Se você rodar o notebook e der erro em células seguintes, pode ser porque a conexão anterior ficou aberta. Sempre feche com `close()`.

In [ ]:
# Exemplo 3: INSERT - Inserindo dados
conexao = sqlite3.connect("escola.db")
cursor = conexao.cursor()

# Inserir um único registro
sql = "INSERT INTO alunos (nome, idade, curso, nota) VALUES (?, ?, ?, ?)"
valores = ("Maria Silva", 20, "Ciência da Computação", 8.75)
cursor.execute(sql, valores)

# Inserir múltiplos registros
varios_alunos = [
    ("João Santos", 22, "Sistemas de Informação", 7.50),
    ("Ana Pereira", 19, "Engenharia de Software", 9.20),
    ("Pedro Costa", 25, "Ciência da Computação", 6.80),
    ("Lúcia Oliveira", 21, "Sistemas de Informação", 8.90),
]
cursor.executemany(sql, varios_alunos)

# SQLite começa em modo autocommit, mas COMMIT explícito é mais seguro
conexao.commit()
print(f"{cursor.rowcount} registros inseridos com sucesso!")

cursor.close()
conexao.close()

In [ ]:
# Exemplo 4: SELECT - Consultando dados
conexao = sqlite3.connect("escola.db")
cursor = conexao.cursor()

cursor.execute("SELECT * FROM alunos")
resultados = cursor.fetchall()

print(f"{'ID':<5s} {'Nome':<20s} {'Idade':<8s} {'Curso':<30s} {'Nota':<8s}")
print("-" * 75)
for linha in resultados:
    print(f"{linha[0]:<5d} {linha[1]:<20s} {linha[2]:<8d} {linha[3]:<30s} {linha[4]:<8.2f}")

cursor.close()
conexao.close()

In [ ]:
# Exemplo 5: SELECT com WHERE, ORDER BY, LIMIT
conexao = sqlite3.connect("escola.db")
cursor = conexao.cursor()

# WHERE - Filtrar por curso
print("1. Alunos de Ciência da Computação:")
cursor.execute("SELECT nome, nota FROM alunos WHERE curso = ?", ("Ciência da Computação",))
for nome, nota in cursor.fetchall():
    print(f"   {nome}: {nota}")

# ORDER BY - Notas em ordem decrescente
print("\n2. Top 3 melhores notas:")
cursor.execute("SELECT nome, nota FROM alunos ORDER BY nota DESC LIMIT 3")
for nome, nota in cursor.fetchall():
    print(f"   {nome}: {nota}")

# WHERE com operadores (AND)
print("\n3. Alunos com nota >= 7.0 e idade > 20:")
cursor.execute("SELECT nome, idade, nota FROM alunos WHERE nota >= ? AND idade > ?", (7.0, 20))
for nome, idade, nota in cursor.fetchall():
    print(f"   {nome} (idade {idade}): {nota}")

# LIMIT com offset (paginação)
print("\n4. Página 1 (2 registros por página):")
cursor.execute("SELECT nome FROM alunos LIMIT 2 OFFSET 0")
for (nome,) in cursor.fetchall():
    print(f"   {nome}")

cursor.close()
conexao.close()

In [ ]:
# Exemplo 6: UPDATE e DELETE
conexao = sqlite3.connect("escola.db")
cursor = conexao.cursor()

# UPDATE - Atualizar nota
cursor.execute("UPDATE alunos SET nota = ? WHERE nome = ?", (9.50, "Maria Silva"))
print(f"Nota atualizada: {cursor.rowcount} registro(s) afetado(s)")

# DELETE - Remover registro
cursor.execute("DELETE FROM alunos WHERE nome = ?", ("Pedro Costa",))
print(f"Registro removido: {cursor.rowcount} registro(s) afetado(s)")

conexao.commit()
cursor.close()
conexao.close()

In [ ]:
# Exemplo 7: JOIN entre tabelas
conexao = sqlite3.connect("escola.db")
cursor = conexao.cursor()

# Criar tabela de cursos
cursor.execute("""
    CREATE TABLE IF NOT EXISTS cursos (
        id INTEGER PRIMARY KEY,
        nome TEXT,
        carga_horaria INTEGER,
        professor TEXT
    )
""")

# Inserir cursos (IGNORE não existe no SQLITE; usamos OR IGNORE)
cursos_data = [
    ("Ciência da Computação", 3600, "Dr. Carlos Mendes"),
    ("Sistemas de Informação", 3200, "Dra. Patrícia Lima"),
    ("Engenharia de Software", 3400, "Dr. Roberto Alves"),
]
for nome, carga, prof in cursos_data:
    cursor.execute(
        "INSERT OR IGNORE INTO cursos (nome, carga_horaria, professor) VALUES (?, ?, ?)",
        (nome, carga, prof)
    )
conexao.commit()

# JOIN: alunos com detalhes do curso
cursor.execute("""
    SELECT a.nome, a.nota, c.carga_horaria, c.professor
    FROM alunos a
    JOIN cursos c ON a.curso = c.nome
    ORDER BY a.nota DESC
""")

print("Alunos com detalhes do curso (JOIN):")
print(f"{'Nome':<20s} {'Nota':<8s} {'Carga Hor.':<12s} {'Professor':<25s}")
print("-" * 65)
for nome, nota, carga, prof in cursor.fetchall():
    print(f"{nome:<20s} {nota:<8.2f} {carga:<12d} {prof:<25s}")

cursor.close()
conexao.close()

In [ ]:
# Exemplo 8: Limpeza — Deletar tabela
conexao = sqlite3.connect("escola.db")
cursor = conexao.cursor()

# Descomente para limpar as tabelas:
# cursor.execute("DROP TABLE IF EXISTS alunos")
# cursor.execute("DROP TABLE IF EXISTS cursos")
# print("Tabelas removidas.")

print("DROP TABLE comentado para segurança. Descomente se quiser limpar.")

cursor.close()
conexao.close()

## ✅ Gabarito dos Exercícios

In [ ]:
# Exercício 1: Inserir e consultar com segurança
def inserir_aluno(nome, idade, curso, nota):
    """Insere um aluno e retorna o ID gerado."""
    conexao = sqlite3.connect("escola.db")
    cursor = conexao.cursor()
    sql = "INSERT INTO alunos (nome, idade, curso, nota) VALUES (?, ?, ?, ?)"
    cursor.execute(sql, (nome, idade, curso, nota))
    conexao.commit()
    id_gerado = cursor.lastrowid
    print(f"Aluno '{nome}' inserido com sucesso! ID: {id_gerado}")
    cursor.close()
    conexao.close()
    return id_gerado

# Teste (descomente):
# inserir_aluno("Teste User", 18, "Ciência da Computação", 8.0)

In [ ]:
# Exercício 2: Relatório de média por curso
conexao = sqlite3.connect("escola.db")
cursor = conexao.cursor()

cursor.execute("""
    SELECT curso,
           COUNT(*) AS total_alunos,
           ROUND(AVG(nota), 2) AS media_nota,
           MIN(nota) AS menor_nota,
           MAX(nota) AS maior_nota
    FROM alunos
    GROUP BY curso
    ORDER BY media_nota DESC
""")

print("Relatório de Notas por Curso:")
print(f"{'Curso':<30s} {'Alunos':<10s} {'Média':<10s} {'Mín':<8s} {'Máx':<8s}")
print("-" * 65)
for curso, total, media, min_n, max_n in cursor.fetchall():
    print(f"{curso:<30s} {total:<10d} {media:<10.2f} {min_n:<8.2f} {max_n:<8.2f}")

cursor.close()
conexao.close()

## 📝 Exercícios Propostos

### Fácil

1. Crie um banco `biblioteca.db` e uma tabela `livros` com colunas: id, titulo, autor, ano, preco.
2. Insira 5 livros na tabela.
3. Consulte todos os livros e exiba formatado.

### Médio

4. Consulte livros com preço maior que R$ 50, ordenados por preço decrescente.
5. Atualize o preço de um livro específico.
6. Delete livros de um autor específico.
7. Use LIMIT e OFFSET para paginar (3 livros por página).
8. Crie uma tabela `autores` e faça JOIN com `livros`.

### Difícil

9. Crie uma função `buscar_livros(termo)` que busca por título ou autor usando LIKE.
10. Crie um sistema de empréstimo: tabela `emprestimos` com id_livro, data_retirada, data_devolucao. Faça JOIN para listar livros emprestados.

## 📌 Resumo

- **`sqlite3`**: módulo nativo do Python — não precisa instalar nada
- **`connect("arquivo.db")**: abre ou cria o banco
- **`cursor.execute(sql, params)`**: executa SQL com `?` como placeholder
- **`commit()`**: salva as alterações
- **`fetchall()`**: busca todos os resultados
- **Parâmetros preparados (`?`)**: previnem SQL Injection
- **SQLite ≠ MySQL/MongoDB**: não precisa de servidor, mas é limitado a um usuário por vez

## 💡 Para Saber Mais

- Se um dia precisar de um banco mais robusto (múltiplos usuários, milhões de acessos), migre para **PostgreSQL** ou **MySQL**
- O SQL que você aprendeu aqui é o mesmo usado nesses bancos — muda só a conexão
- Bibliotecas como **SQLAlchemy** permitem trocar de banco sem mudar o código Python

## ⚠️ Erros Comuns

| Erro | Exemplo | Correção |
|------|---------|----------|
| SQL Injection | `f"SELECT * FROM users WHERE nome='{nome}'"` | Use parâmetros: `cursor.execute(..., (nome,))` |
| Esquecer `?` | `cursor.execute("INSERT INTO ... VALUES (nome)")` | Use `VALUES (?)` e passe `(nome,)` |
| Vírgula faltando na tupla | `(nome)` em vez de `(nome,)` | Tupla de 1 elemento precisa da vírgula |
| Conexão não fechada | Notebook trava, arquivo .db fica bloqueado | Sempre feche com `close()` |

---

[◀ Anterior](33_DSA_Busca_Ordenacao.ipynb) | [📖 Índice](00_Index.ipynb) | [Próximo ▶](35_MongoDB.ipynb)